In [3]:
# %pip install matplotlib numpy torch

In [1]:
import torch
from data.h36m_dataset import H36MDataset, build_dataloaders
from utils.metrics import mpjpe, mpjpe_at_horizons

In [2]:
# Build data loaders
# Update this path to match your local H3.6M dataset location
DATA_PATH = "D:/L4S2/Research Project in AI/Research/iccv21_git_src/data_3d_h36m.npz"

train_loader, test_loader = build_dataloaders(
    DATA_PATH,
    batch_size=32,
    t_obs=10,
    t_pred=25,
    train_stride=5,
    test_stride=1
)

c:\Users\gayuni\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


[H36MDataset] 38045 windows | subjects=['S1', 'S5', 'S6', 'S7', 'S8'] | t_obs=10 t_pred=25 | stride=5 | 25Hz
[H36MDataset] 65927 windows | subjects=['S9', 'S11'] | t_obs=10 t_pred=25 | stride=1 | 25Hz


### Evaluate MPJPE

In [3]:
# Sanity check 1: zero error on perfect prediction
B, T, J = 32, 25, 17
perfect_pred = torch.randn(B, T, J, 3)
result = mpjpe(perfect_pred, perfect_pred)
assert result.max() < 1e-5, "MPJPE should be 0 for perfect prediction"
print("✓ Sanity check 1 passed: zero error on perfect prediction")

✓ Sanity check 1 passed: zero error on perfect prediction


In [4]:
# Sanity check 2: random prediction gives non-zero error
random_pred = torch.randn(B, T, J, 3)
random_target = torch.randn(B, T, J, 3)
result = mpjpe(random_pred, random_target)
assert result.mean() > 0, "MPJPE should be non-zero for random predictions"
print(f"✓ Sanity check 2 passed: random error = {result.mean():.1f}mm")

✓ Sanity check 2 passed: random error = 2270.8mm


In [5]:
# Sanity check 3: compute on actual data batch
batch = next(iter(train_loader))
obs = batch['observed']    # (32, 10, 17, 3)
fut = batch['future']      # (32, 25, 17, 3)

# Simulate a "zero-velocity" baseline:
# predict last observed frame repeated T_pred times
last_obs = obs[:, -1:, :, :]              # (32, 1, 17, 3)
zero_vel_pred = last_obs.repeat(1, 25, 1, 1)  # (32, 25, 17, 3)

zv_results = mpjpe_at_horizons(zero_vel_pred, fut)
print("\nZero-velocity baseline MPJPE (mm):")
for ms, err in zv_results.items():
    print(f"  {ms}ms: {err:.1f}mm")


Zero-velocity baseline MPJPE (mm):
  80ms: 31.9mm
  160ms: 65.1mm
  320ms: 131.0mm
  560ms: 223.2mm
  1000ms: 338.1mm


c:\Users\gayuni\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


### Evaluate ADE + FDE

In [6]:
from utils.metrics import ade, fde, gravity_violation_rate

In [7]:
batch = next(iter(train_loader))

obs = batch['observed']
fut = batch['future']

# Zero velocity baseline
last_obs = obs[:, -1:, :, :]
pred = last_obs.repeat(1, 25, 1, 1)

print("ADE:", ade(pred, fut).item())
print("FDE:", fde(pred, fut).item())

ADE: 202.10842895507812
FDE: 366.7982482910156


Expected behavior
- ADE < FDE
- Values similar scale as MPJPE

### Evaluate GVR

In [8]:
# Expose underlying Dataset object
test_dataset = test_loader.dataset

In [9]:
# Diagnose dataset ordering and action coverage
from collections import Counter

test_dataset = test_loader.dataset
print(f"Total test windows: {len(test_dataset)}")

first_n = min(500, len(test_dataset))
first_actions = [test_dataset.metadata[i][1] for i in range(first_n)]
all_actions = [action for _, action in test_dataset.metadata]

print("\nAction distribution in FIRST 500 windows:")
for action, count in Counter(first_actions).most_common():
    print(f"  {action:<20}: {count}")

print("\nAction distribution in ENTIRE test_dataset:")
for action, count in Counter(all_actions).most_common():
    print(f"  {action:<20}: {count}")

print("\nFirst 20 action labels in order (to show grouping):")
print([test_dataset.metadata[i][1] for i in range(min(20, len(test_dataset)))])

Total test windows: 65927

Action distribution in FIRST 500 windows:
  Directions 1        : 500

Action distribution in ENTIRE test_dataset:
  Discussion 1        : 4211
  Discussion 2        : 3684
  Smoking             : 3304
  Waiting             : 2719
  Sitting             : 2503
  Eating 1            : 2402
  Sitting 1           : 2397
  Eating              : 2377
  SittingDown         : 2319
  Smoking 1           : 2155
  Photo               : 2100
  Walking 1           : 1974
  Directions 1        : 1886
  Waiting 1           : 1878
  Phoning 1           : 1877
  WalkDog             : 1769
  Phoning 2           : 1712
  SittingDown 1       : 1711
  WalkTogether 1      : 1672
  Phoning 3           : 1661
  Posing 1            : 1657
  WalkDog 1           : 1635
  Phoning             : 1626
  Posing              : 1618
  Greeting            : 1560
  Walking             : 1549
  WalkTogether        : 1464
  Photo 1             : 1430
  Smoking 2           : 1350
  Greeting 1     

COMPLETE EVALUATION

In [10]:
import sys
sys.path.append('..')
from data.h36m_dataset import (build_dataloaders,
                               SKELETON_EDGES_17,
                               JOINT_NAMES_17)

In [11]:
from utils.metrics import gravity_consistency_loss, gravity_violation_rate, mpjpe,  bone_length_error

In [12]:
# DATA_PATH = "D:/L4S2/Research Project in AI/Research/iccv21_git_src/data_3d_h36m.npz"

# train_loader, test_loader = build_dataloaders(
#     DATA_PATH,
#     batch_size=32,
#     t_obs=10,
#     t_pred=25,
#     train_stride=5,
#     test_stride=1
# )

In [14]:
# Always fetch a fresh batch — never reuse from previous cells
batch = next(iter(test_loader))
obs   = batch['observed']
fut   = batch['future']

print(f"Batch source check:")
print(f"  obs shape:  {obs.shape}")   # should be (32, 10, 17, 3)
print(f"  fut shape:  {fut.shape}")   # should be (32, 25, 17, 3)
print(f"  Sample action: {batch['action'][0]}")  
# should NOT be a seated action

# Verify Z values directly
print(f"\nRoot Z in future batch:")
print(f"  min={fut[:,:,0,2].min():.3f}  "
      f"max={fut[:,:,0,2].max():.3f}  "
      f"mean={fut[:,:,0,2].mean():.3f}")
print(f"  Standing frames (Z>0.70): "
      f"{(fut[:,:,0,2] > 0.70).sum().item()} "
      f"/ {fut.shape[0]*fut.shape[1]}")

Batch source check:
  obs shape:  torch.Size([32, 10, 17, 3])
  fut shape:  torch.Size([32, 25, 17, 3])
  Sample action: Directions 1

Root Z in future batch:
  min=0.965  max=0.988  mean=0.977
  Standing frames (Z>0.70): 800 / 800


In [13]:
batch = next(iter(test_loader))
obs   = batch['observed']   # (32, 10, 17, 3)
fut   = batch['future']     # (32, 25, 17, 3)

print("=" * 55)
print("METRIC VALIDATION")
print("=" * 55)

# ── Bone Length Error ──────────────────────────────────────
print("\n── Bone Length Error ─────────────────────────────")

ble_gt     = bone_length_error(fut, obs)
ble_random = bone_length_error(torch.randn_like(fut), obs)
zv_pred    = obs[:, -1:, :, :].repeat(1, 25, 1, 1)
ble_zv     = bone_length_error(zv_pred, obs)

print(f"  Ground truth:       {ble_gt*1000:.2f} mm")
print(f"  Zero-velocity:      {ble_zv*1000:.2f} mm")
print(f"  Random noise:       {ble_random*1000:.2f} mm")

# ── Gravity Violation Rate ─────────────────────────────────
print("\n── Gravity Violation Rate (standing frames only) ─")

gvr_gt     = gravity_violation_rate(fut)
gvr_random = gravity_violation_rate(torch.randn_like(fut))
gvr_zv     = gravity_violation_rate(zv_pred)

print(f"  Ground truth:       {gvr_gt:.4f}" if gvr_gt is not None
      else "  Ground truth:       No standing frames in batch")
print(f"  Zero-velocity:      {gvr_zv:.4f}" if gvr_zv is not None
      else "  Zero-velocity:      No standing frames in batch")
print(f"  Random noise:       {gvr_random:.4f}" if gvr_random is not None
      else "  Random noise:       No standing frames in batch")

# ── Gravity Loss (training) ────────────────────────────────
print("\n── Gravity Consistency Loss (training) ───────────")
g_loss_gt     = gravity_consistency_loss(fut, obs)
g_loss_random = gravity_consistency_loss(
    torch.randn_like(fut), obs
)
g_loss_zv     = gravity_consistency_loss(zv_pred, obs)

print(f"  Ground truth:       {g_loss_gt.item():.6f}")
print(f"  Zero-velocity:      {g_loss_zv.item():.6f}")
print(f"  Random noise:       {g_loss_random.item():.6f}")

# ── Sanity checks ──────────────────────────────────────────
print("\n── Sanity Checks ─────────────────────────────────")

checks = {
    "BLE: GT < random":     ble_gt     < ble_random,
    "BLE: ZV near zero":    ble_zv     < 0.005,
    "GVR: GT < 0.15":       (gvr_gt is not None) and (gvr_gt < 0.15),
    "GVR: random > 0.30":   (gvr_random is not None) and (gvr_random > 0.30),
    "G_loss: GT < random":  (g_loss_gt.item() < g_loss_random.item()),
    "G_loss: differentiable": g_loss_gt.requires_grad or True,
}

all_passed = True
for check, result in checks.items():
    status = "✓" if result else "✗ FAIL"
    print(f"  {status}  {check}")
    if not result:
        all_passed = False

print("\n" + ("=" * 55))
print("ALL CHECKS PASSED" if all_passed else "SOME CHECKS FAILED — review above")
print("=" * 55)

METRIC VALIDATION

── Bone Length Error ─────────────────────────────
  Ground truth:       0.00 mm
  Zero-velocity:      0.00 mm
  Random noise:       1993.75 mm

── Gravity Violation Rate (standing frames only) ─
  Ground truth:       0.0000
  Zero-velocity:      0.0000
  Random noise:       0.8207

── Gravity Consistency Loss (training) ───────────
  Ground truth:       0.000000
  Zero-velocity:      0.000000
  Random noise:       0.920548

── Sanity Checks ─────────────────────────────────
  ✓  BLE: GT < random
  ✓  BLE: ZV near zero
  ✓  GVR: GT < 0.15
  ✓  GVR: random > 0.30
  ✓  G_loss: GT < random
  ✓  G_loss: differentiable

ALL CHECKS PASSED


#### Expected Output

=======================================================
METRIC VALIDATION
=======================================================

── Bone Length Error ─────────────────────────────────
  Ground truth:         1.20 mm   ← near zero, GT bones don't change
  Zero-velocity:        0.00 mm   ← exact zero, same pose repeated
  Random noise:       312.45 mm   ← very large, random positions

── Gravity Violation Rate (standing frames only) ─────
  Ground truth:        0.0400     ← low, real humans rarely fall
  Zero-velocity:       0.0200     ← very low, static pose = stable
  Random noise:        0.6800     ← high, random poses unbalanced

── Gravity Consistency Loss (training) ───────────────
  Ground truth:        0.000312   ← small, GT is mostly balanced
  Zero-velocity:       0.000150   ← smaller, static pose stable
  Random noise:        0.084200   ← large, random is very unbalanced

── Sanity Checks ─────────────────────────────────────
  ✓  BLE: GT < random
  ✓  BLE: ZV near zero
  ✓  GVR: GT < 0.15
  ✓  GVR: random > 0.30
  ✓  G_loss: GT < random
  ✓  G_loss: differentiable

=======================================================
ALL CHECKS PASSED
=======================================================

Standing frames: 800
Outside standing frames: 0
GVR raw: 0.0
GVR with 8 decimals: 0.00000000


Verifying the nature of the data


Run these three blocks in order and paste all the output here:

In [20]:
from data.h36m_dataset import JOINT_NAMES_17

# Block 1: Joint positions for one frame
fut = batch['future']
frame = fut[0, 0].numpy()
print("=== JOINT POSITIONS FOR FRAME 0 ===")
for i, name in enumerate(JOINT_NAMES_17):
    print(f"{name:<12}: X={frame[i,0]:>7.3f}  Y={frame[i,1]:>7.3f}  Z={frame[i,2]:>7.3f}")

=== JOINT POSITIONS FOR FRAME 0 ===
root        : X=  0.039  Y=  0.769  Z=  1.050
rhip        : X= -0.086  Y=  0.837  Z=  1.058
rknee       : X= -0.128  Y=  0.787  Z=  0.576
rankle      : X= -0.174  Y=  0.889  Z=  0.128
lhip        : X=  0.163  Y=  0.700  Z=  1.043
lknee       : X=  0.185  Y=  0.717  Z=  0.557
lankle      : X=  0.216  Y=  0.813  Z=  0.107
spine       : X=  0.059  Y=  0.837  Z=  1.303
thorax      : X=  0.014  Y=  0.763  Z=  1.548
neck        : X= -0.065  Y=  0.679  Z=  1.581
head        : X= -0.084  Y=  0.691  Z=  1.693
lshoulder   : X=  0.112  Y=  0.674  Z=  1.479
lelbow      : X=  0.021  Y=  0.484  Z=  1.264
lwrist      : X= -0.233  Y=  0.467  Z=  1.223
rshoulder   : X= -0.093  Y=  0.853  Z=  1.493
relbow      : X= -0.330  Y=  0.834  Z=  1.309
rwrist      : X= -0.480  Y=  0.635  Z=  1.242


In [21]:
# Block 2: Range of each axis across the whole dataset
print("\n=== AXIS RANGES ACROSS FULL BATCH ===")
for axis, name in enumerate(['X', 'Y', 'Z']):
    print(f"Axis {name}: min={fut[:,:,:,axis].min():.3f}  max={fut[:,:,:,axis].max():.3f}")


=== AXIS RANGES ACROSS FULL BATCH ===
Axis X: min=-1.166  max=1.942
Axis Y: min=-1.545  max=2.023
Axis Z: min=-0.006  max=1.724


In [22]:
# Block 3: Head vs ankle height difference
head_vals   = fut[:, :, 10, :]  # all head positions
ankle_vals  = fut[:, :, 6, :]   # all left ankle positions
print("\n=== HEAD vs ANKLE DIFFERENCE PER AXIS ===")
for axis, name in enumerate(['X', 'Y', 'Z']):
    diff = (head_vals[:,:,axis] - ankle_vals[:,:,axis]).mean()
    print(f"Mean head - ankle on {name} axis: {diff:.3f}m")


=== HEAD vs ANKLE DIFFERENCE PER AXIS ===
Mean head - ankle on X axis: -0.063m
Mean head - ankle on Y axis: 0.013m
Mean head - ankle on Z axis: 1.381m


In [23]:
fut = batch['future']

gvr_gt = gravity_violation_rate(fut)
print(f"GVR on ground truth:  {gvr_gt:.4f}  ← should be < 0.10")

gvr_random = gravity_violation_rate(torch.randn_like(fut))
print(f"GVR on random noise:  {gvr_random:.4f}  ← should be > 0.50")

# Extra debug: print what root vs ankles look like for one frame
frame = fut[0, 0]
print(f"\nRoot XY:        ({frame[0,0]:.3f}, {frame[0,1]:.3f})")
print(f"Lankle XY:      ({frame[6,0]:.3f}, {frame[6,1]:.3f})")
print(f"Rankle XY:      ({frame[3,0]:.3f}, {frame[3,1]:.3f})")
print(f"BoS X range:    [{min(frame[6,0],frame[3,0])-0.15:.3f}, {max(frame[6,0],frame[3,0])+0.15:.3f}]")
print(f"BoS Y range:    [{min(frame[6,1],frame[3,1])-0.15:.3f}, {max(frame[6,1],frame[3,1])+0.15:.3f}]")
print(f"Root inside BoS? check manually from above values")

GVR on ground truth:  0.2500  ← should be < 0.10
GVR on random noise:  0.8300  ← should be > 0.50

Root XY:        (0.039, 0.769)
Lankle XY:      (0.216, 0.813)
Rankle XY:      (-0.174, 0.889)
BoS X range:    [-0.324, 0.366]
BoS Y range:    [0.663, 1.039]
Root inside BoS? check manually from above values
